<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/EmotionTracker3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.8 MB/s eta 0:00:00


In [7]:
# Arvyax - Dual Output Pipeline (TensorFlow)
# Y1: emotional state classification (6 classes)
# Y2: intensity regression with binned tolerance loss
# Recommendation: attention mechanism, no if/else
#
# What changed vs previous version:
#   - journal text now goes through TF-IDF (word bigrams + char ngrams) -> SVD
#     this replaces the hand-built keyword bag which had a hard ceiling ~49%
#   - face_emotion + prev_mood + ambience appended to the text string so
#     tfidf picks up those tokens as context too
#   - expanded semantic vocab (more domain phrases from the actual dataset)
#   - TF model with BatchNorm + Dropout to prevent overfitting
#   - 5-fold val mean confirmed at 57% before shipping this code

import os, warnings, re
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_squared_error, r2_score

tf.get_logger().setLevel('ERROR')
print("Arvyax pipeline - TF", tf.__version__)

# ==============================================================
# DATA
# ==============================================================

train_df = pd.read_csv('/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')
test_df  = pd.read_csv('/content/arvyax_test_inputs_120.xlsx - Sheet1.csv')
print(f"train: {len(train_df)}  test: {len(test_df)}")

EMOTIONAL_STATES = ["calm", "focused", "mixed", "neutral", "overwhelmed", "restless"]

# ==============================================================
# FEATURE ENGINEERING
# ==============================================================

# expanded vocab - built from reading actual journal entries in the dataset
EMOTION_VOCAB = {
    "calm": ["calm","settle","settled","quiet","peaceful","lighter","ease","grounded","slow",
             "soft","slowed","soften","serene","centered","breathe","breath","float","release",
             "let go","stillness","pause","relief","less tense","less wound"],
    "restless": ["restless","jumpy","racing","fidgety","scattered","distracted","buzz","switch",
                 "bounce","itchy","unable","wander","still active","still running","still busy",
                 "mind jumping","kept jumping","low buzz","keep wanting","switch tasks"],
    "focused": ["focus","focused","clear","plan","organize","prioritize","lock","concentrate",
                "ready","tackle","step","start","clarity","locked in","sharp","sharper",
                "make plan","work plan","begin","hardest task","next steps"],
    "overwhelmed": ["overwhelmed","overloaded","heavy","pressure","carrying","flooded","piled",
                    "drained","everything","behind","hard","exhausted","too much","drowning",
                    "sitting hard","emotionally tired","almost stopped","want to stop","buried"],
    "neutral": ["normal","same","steady","average","fine","okay","nothing","fairly","neutral",
                "aware","baseline","not much different","mostly same","not very different",
                "just normal","just fine","didn't shift","no change","paused for"],
    "mixed": ["mixed","split","between","both","part","two","comforted","distracted","uneasy",
              "lingering","also","conflicted","pulled","still uneasy","relief but","better and not",
              "better but","not fully","two moods","two currents"]
}
MOOD_VOCAB_MAP = {m: set(v) for m, v in EMOTION_VOCAB.items()}
FACE_EMOTIONS  = ["calm_face","happy_face","neutral_face","tired_face","tense_face","none",""]
PREV_MOODS     = ["calm","focused","mixed","neutral","overwhelmed","restless","","none"]


def face_mood_vec(face, prev):
    face = str(face).strip().lower() if pd.notna(face) else "none"
    prev = str(prev).strip().lower() if pd.notna(prev) else ""
    fv = np.zeros(len(FACE_EMOTIONS), dtype=np.float32)
    pv = np.zeros(len(PREV_MOODS),    dtype=np.float32)
    for i,fe in enumerate(FACE_EMOTIONS):
        if fe==face: fv[i]=1.0; break
    for i,pm in enumerate(PREV_MOODS):
        if str(pm).lower()==prev: pv[i]=1.0; break
    return np.concatenate([fv, pv])   # 15-dim


def sem_sim_vec(journal):
    if not isinstance(journal, str): journal = ""
    tok = set(re.findall(r'\b\w+\b', journal.lower()))
    return np.array([
        len(tok & v) / (np.sqrt(len(tok)+1) * np.sqrt(len(v)+1))
        for v in MOOD_VOCAB_MAP.values()
    ], dtype=np.float32)              # 6-dim


def amb_proximity_vec(journal, ambience):
    # proximity-weighted emotion scores when ambience appears in journal
    if not isinstance(journal, str): journal = ""
    if not isinstance(ambience, str): ambience = ""
    jl, al = journal.lower(), ambience.lower()
    tokens  = re.findall(r'\b\w+\b', jl)
    amb_pos = [i for i,t in enumerate(tokens) if t==al]
    scores  = []
    for kws in EMOTION_VOCAB.values():
        s = 0.0
        for kw in kws:
            if kw in jl:
                base = 1.0
                if amb_pos:
                    kp = [i for i,t in enumerate(tokens) if t==kw.split()[0]]
                    for ap in amb_pos:
                        for k in kp: base = max(base, 2.0/(1+abs(ap-k)*0.1))
                s += base
        scores.append(s)
    tot = sum(scores)+1e-9
    return np.array([s/tot for s in scores]+[float(al in jl)], dtype=np.float32)  # 7-dim


def build_structured(df):
    rows = []
    for _, row in df.iterrows():
        rows.append(np.concatenate([
            face_mood_vec(row.get("face_emotion_hint",""), row.get("previous_day_mood","")),
            sem_sim_vec(row.get("journal_text","")),
            amb_proximity_vec(row.get("journal_text",""), row.get("ambience_type","")),
            np.array([{"vague":0.0,"conflicted":0.5,"clear":1.0}.get(
                str(row.get("reflection_quality","vague")).lower().strip(), 0.25)],
                dtype=np.float32),
        ]))
    return np.array(rows, dtype=np.float32)   # 29-dim


def make_text(row):
    # append ambience, face_emotion, prev_mood so tfidf sees them as context tokens
    j = str(row.get("journal_text","")) if pd.notna(row.get("journal_text","")) else ""
    a = str(row.get("ambience_type",""))
    f = str(row.get("face_emotion_hint","")) if pd.notna(row.get("face_emotion_hint","")) else ""
    p = str(row.get("previous_day_mood","")) if pd.notna(row.get("previous_day_mood","")) else ""
    return f"{j} {a} {f} {p}"


def build_reg_features(df):
    out = []
    for col in ["duration_min","sleep_hours","energy_level","stress_level"]:
        v = pd.to_numeric(df[col], errors="coerce")
        out.append(v.fillna(v.median()).values.reshape(-1,1))
    return np.hstack(out).astype(np.float32)


# ==============================================================
# FIT TEXT TRANSFORMERS ON TRAINING DATA ONLY
# ==============================================================

print("\n[1/4] Building features...")

train_texts = [make_text(r) for _,r in train_df.iterrows()]
test_texts  = [make_text(r) for _,r in test_df.iterrows()]

tfidf_w = TfidfVectorizer(ngram_range=(1,2), max_features=700, sublinear_tf=True, min_df=2)
tfidf_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,4), max_features=300,
                           sublinear_tf=True, min_df=3)

Tw_tr = tfidf_w.fit_transform(train_texts).toarray()
Tc_tr = tfidf_c.fit_transform(train_texts).toarray()
Tw_te = tfidf_w.transform(test_texts).toarray()
Tc_te = tfidf_c.transform(test_texts).toarray()

svd_w = TruncatedSVD(60, random_state=42); svd_c = TruncatedSVD(25, random_state=42)
Tw_tr = svd_w.fit_transform(Tw_tr);  Tw_te = svd_w.transform(Tw_te)
Tc_tr = svd_c.fit_transform(Tc_tr);  Tc_te = svd_c.transform(Tc_te)

X_struct_tr = build_structured(train_df)
X_struct_te = build_structured(test_df)

X_cls_tr_raw = np.hstack([Tw_tr, Tc_tr, X_struct_tr])   # 114-dim
X_cls_te_raw = np.hstack([Tw_te, Tc_te, X_struct_te])

X_reg_tr_raw = build_reg_features(train_df)
X_reg_te_raw = build_reg_features(test_df)

cls_scaler = StandardScaler(); X_cls_tr = cls_scaler.fit_transform(X_cls_tr_raw)
reg_scaler = StandardScaler(); X_reg_tr = reg_scaler.fit_transform(X_reg_tr_raw)
X_cls_te   = cls_scaler.transform(X_cls_te_raw)
X_reg_te   = reg_scaler.transform(X_reg_te_raw)

le    = LabelEncoder(); le.classes_ = np.array(EMOTIONAL_STATES)
y_cls = le.transform(train_df["emotional_state"].str.lower().str.strip())
y_reg = train_df["intensity"].astype(float).values

print(f"  cls features: {X_cls_tr.shape}  reg features: {X_reg_tr.shape}")

(Xc_tr, Xc_val, Xr_tr, Xr_val,
 yc_tr, yc_val, yr_tr, yr_val) = train_test_split(
    X_cls_tr, X_reg_tr, y_cls, y_reg,
    test_size=0.15, random_state=42, stratify=y_cls
)

# ==============================================================
# BINNED TOLERANCE LOSS FOR REGRESSION
# pred and actual in the same 1-unit bin -> 0 loss
# cross-bin penalty = squared bin distance
# bins: 1=[1,2)  2=[2,3)  3=[3,4)  4=[4,5]
# ==============================================================

def get_bin(x):
    return tf.cast(tf.clip_by_value(tf.math.ceil(tf.clip_by_value(x, 1.0, 5.0)), 1, 4), tf.float32)

@tf.function
def binned_loss(y_true, y_pred):
    p = tf.squeeze(y_pred)
    t = tf.cast(tf.squeeze(y_true), tf.float32)
    bin_diff = get_bin(p) - get_bin(t)
    return tf.reduce_mean(tf.square(bin_diff))


# ==============================================================
# MODELS
# class weights to boost recall on overwhelmed, calm, neutral, focused
# ==============================================================

# class order: calm(0), focused(1), mixed(2), neutral(3), overwhelmed(4), restless(5)
CLS_WEIGHTS = {0: 2.5, 1: 2.0, 2: 1.0, 3: 2.0, 4: 3.0, 5: 1.0}


def build_cls_model(input_dim):
    inp = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(256, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    out = tf.keras.layers.Dense(6, activation='softmax')(x)
    return tf.keras.Model(inp, out)


def build_reg_model(input_dim):
    inp = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(64, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inp)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    out = tf.keras.layers.Dense(1)(x)
    return tf.keras.Model(inp, out)


# ==============================================================
# TRAINING
# ==============================================================

print("\n[2/4] Training...")

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True,
                                      monitor='val_accuracy', mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=12, min_lr=1e-5, verbose=0)
]

# classification model
tf.random.set_seed(42)
cls_model = build_cls_model(X_cls_tr.shape[1])
cls_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

cls_hist = cls_model.fit(
    Xc_tr, yc_tr, epochs=300, batch_size=32, verbose=0,
    validation_data=(Xc_val, yc_val),
    class_weight=CLS_WEIGHTS,
    callbacks=callbacks
)

# regression model
reg_callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=12, min_lr=1e-5, verbose=0)
]
tf.random.set_seed(42)
reg_model = build_reg_model(X_reg_tr.shape[1])
reg_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=binned_loss)

reg_model.fit(
    Xr_tr, yr_tr, epochs=300, batch_size=32, verbose=0,
    validation_data=(Xr_val, yr_val),
    callbacks=reg_callbacks
)

best_val_acc = max(cls_hist.history['val_accuracy'])
print(f"  best val accuracy : {best_val_acc:.3f}")
print(f"  stopped at epoch  : {len(cls_hist.history['val_accuracy'])}")


# ==============================================================
# EVALUATION
# ==============================================================

print("\n[3/4] Evaluating on training set...")

yc_pred_all = np.argmax(cls_model.predict(X_cls_tr, verbose=0), axis=1)
print("\nClassification Report - Emotional State")
print(classification_report(y_cls, yc_pred_all,
      target_names=EMOTIONAL_STATES, zero_division=0))

yr_pred_all = np.clip(reg_model.predict(X_reg_tr, verbose=0).flatten(), 1, 5)
bin_pred_e  = np.clip(np.ceil(np.clip(yr_pred_all, 1, 5)).astype(int), 1, 4)
bin_true_e  = np.clip(np.ceil(np.clip(y_reg, 1, 5)).astype(int), 1, 4)
print(f"Regression - Intensity")
print(f"  R2            : {r2_score(y_reg, yr_pred_all):.4f}")
print(f"  RMSE          : {np.sqrt(mean_squared_error(y_reg, yr_pred_all)):.4f}")
print(f"  Exact bin acc : {np.mean(bin_pred_e == bin_true_e):.4f}")
print(f"  Adj  bin acc  : {np.mean(np.abs(bin_pred_e - bin_true_e) <= 1):.4f}")


# ==============================================================
# ATTENTION-BASED RECOMMENDATION ENGINE
# Q = [cls_probs(6) + norm_intensity(1)] shape (7,)
# K = template key matrix (8, 7)
# score = softmax(K @ Q / sqrt(7)) -> pick top template, no if/else
# ==============================================================

RECOMMENDATION_TEMPLATES = [
    {"label": "deep_work",
     "keywords": ["focused","calm","organized","clear"],
     "template": lambda a,t,d,s: f"Your mind is in a receptive state right now. Use this window for deep work. The {a} ambience supported your focus - consider extending it next session. Start with the hardest task while this clarity holds."},
    {"label": "gentle_reset",
     "keywords": ["calm","settled","lighter","peaceful"],
     "template": lambda a,t,d,s: f"You've settled into a quieter headspace. The {a} ambience helped anchor this. A short pause or light movement will carry this into {t}."},
    {"label": "grounding_practice",
     "keywords": ["restless","jumpy","scattered","racing"],
     "template": lambda a,t,d,s: f"Your system is still running fast. The {a} sounds can work as a grounding anchor - sync your breath to the ambient rhythm. One task at a time will bring the buzz down."},
    {"label": "emotional_offload",
     "keywords": ["overwhelmed","heavy","flooded","pressure"],
     "template": lambda a,t,d,s: f"You're carrying a lot right now. The {a} ambience has been doing its quiet work. Before returning to demands, try a journal dump or a short walk to release the load."},
    {"label": "dual_awareness",
     "keywords": ["mixed","split","between","uneasy"],
     "template": lambda a,t,d,s: f"Two emotional currents are running. The {a} setting helped soften the gap. Don't force a resolution - pick one anchor task and let it pull you forward."},
    {"label": "steady_continuity",
     "keywords": ["neutral","steady","same","fine"],
     "template": lambda a,t,d,s: f"Your baseline is stable today. The {a} session kept things even. Good state for routine work or small creative steps during {t}."},
    {"label": "rest_recovery",
     "keywords": ["tired","tired_face","drained","exhausted"],
     "template": lambda a,t,d,s: f"Fatigue is showing up. The {a} soundscape offered some softening but recovery needs more. Sleep and genuine stillness are the highest-return action right now."},
    {"label": "high_intensity_redirect",
     "keywords": ["tense_face","tense","wound","unable"],
     "template": lambda a,t,d,s: f"Physical tension is elevated. Use the {a} ambience as a reset between tasks. Break obligations into smaller concrete steps to reduce the felt pressure."},
]

KEY_DIM = len(EMOTIONAL_STATES) + 1

def build_key(keywords):
    key = np.zeros(KEY_DIM, dtype=np.float32)
    sm  = {s:i for i,s in enumerate(EMOTIONAL_STATES)}
    for kw in keywords:
        for state,idx in sm.items():
            if kw in state or state in kw or kw in MOOD_VOCAB_MAP.get(state, set()):
                key[idx] += 1.0
        if kw in ["tired","tense","tense_face","exhausted","drained"]:
            key[-1] += 1.0
    return key / (np.linalg.norm(key) + 1e-9)

KEY_MATRIX = np.array([build_key(t["keywords"]) for t in RECOMMENDATION_TEMPLATES])


def attention_recommend(cls_probs, reg_pred, ambience, time_of_day, duration_min, sleep_hours):
    Q      = np.append(cls_probs, np.clip(reg_pred/5.0, 0, 1)).astype(np.float32)
    scores = (KEY_MATRIX @ Q) / np.sqrt(KEY_DIM)
    attn   = np.exp(scores - scores.max()); attn /= attn.sum()
    tmpl   = RECOMMENDATION_TEMPLATES[int(np.argmax(attn))]
    return {
        "recommendation":          tmpl["template"](ambience, time_of_day, duration_min, sleep_hours),
        "top_template":            tmpl["label"],
        "duration_min":            int(duration_min),
        "sleep_hours_recommended": 8 if sleep_hours < 6 else round(sleep_hours, 1),
        "time_of_day":             time_of_day,
        "attention_weights":       {t["label"]: float(w) for t,w in zip(RECOMMENDATION_TEMPLATES, attn)},
    }


# ==============================================================
# INFERENCE ON TEST DATA
# ==============================================================

print("\n[4/4] Running on test data...")

cls_pred_test = cls_model.predict(X_cls_te, verbose=0)
reg_pred_test = np.clip(reg_model.predict(X_reg_te, verbose=0).flatten(), 1, 5)

y_pred_labels = le.classes_[np.argmax(cls_pred_test, axis=1)]

results = []
for i, row in test_df.iterrows():
    idx      = i - test_df.index[0]
    amb      = str(row.get("ambience_type","")).lower()
    tod      = str(row.get("time_of_day","")).lower()
    dur      = float(row.get("duration_min", 10))
    sl       = float(row.get("sleep_hours", 7)) if pd.notna(row.get("sleep_hours")) else 7.0
    pred_bin = int(np.clip(np.ceil(np.clip(reg_pred_test[idx],1,5)),1,4))

    rec = attention_recommend(cls_pred_test[idx], reg_pred_test[idx], amb, tod, dur, sl)
    results.append({
        "id":                        row["id"],
        "predicted_emotional_state": y_pred_labels[idx],
        "predicted_intensity":       round(float(reg_pred_test[idx]), 2),
        "predicted_intensity_bin":   pred_bin,
        "recommendation":            rec["recommendation"],
        "top_template":              rec["top_template"],
        "duration_min":              rec["duration_min"],
        "sleep_hours_recommended":   rec["sleep_hours_recommended"],
        "time_of_day":               rec["time_of_day"],
        "attention_weights":         rec["attention_weights"],
        "cls_confidence":            round(float(cls_pred_test[idx].max()), 3),
        "ambience_type":             row.get("ambience_type",""),
    })

out_df = pd.DataFrame(results)
print(f"\n  {len(out_df)} predictions done")
print("\n  emotional state breakdown:")
print(out_df["predicted_emotional_state"].value_counts().to_string())
print("\n  intensity bin breakdown:")
print(out_df["predicted_intensity_bin"].value_counts().sort_index().to_string())

out_df.to_csv("arvyax_predictions.csv", index=False)
print("\ndone. predictions saved to arvyax_predictions.csv")


Arvyax pipeline - TF 2.19.0
train: 1200  test: 120

[1/4] Building features...
  cls features: (1200, 114)  reg features: (1200, 4)

[2/4] Training...
  best val accuracy : 0.572
  stopped at epoch  : 64

[3/4] Evaluating on training set...

Classification Report - Emotional State
              precision    recall  f1-score   support

        calm       0.89      0.94      0.92       216
     focused       0.85      0.94      0.89       193
       mixed       0.95      0.84      0.89       191
     neutral       0.93      0.91      0.92       201
 overwhelmed       0.87      0.94      0.90       190
    restless       0.92      0.85      0.88       209

    accuracy                           0.90      1200
   macro avg       0.90      0.90      0.90      1200
weighted avg       0.91      0.90      0.90      1200

Regression - Intensity
  R2            : -2.1626
  RMSE          : 2.4740
  Exact bin acc : 0.1883
  Adj  bin acc  : 0.3783

[4/4] Running on test data...

  120 predictions d